# ORTHRUS — run a full vulnerability scan in Google Colab

This notebook installs [ORTHRUS](https://github.com/ankitjha67/orthrus), starts the
**bundled, deliberately-vulnerable practice target** (bound to `127.0.0.1` only), runs the
full recon → scan → confirm → report pipeline against it, and prints the findings.

> ## ⚠️ Authorized testing only
> ORTHRUS sends **real attack payloads** and actively tries to exploit findings. Only point it
> at systems you **own** or have **explicit written permission** to test. This notebook scans
> the bundled local practice app, which is the safe thing to learn on. Do **not** change the
> target to a system you are not authorized to test.

**How to use:** `Runtime → Run all`, or run each cell top to bottom.

## 1. Install ORTHRUS
Clone the repo and install the lean core (pure-Python; no system binaries needed).

In [ ]:
import os
REPO = "/content/orthrus"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/ankitjha67/orthrus.git {REPO}
os.chdir(REPO)
!pip -q install -e .
print("\n✅ installed —", end=" ")
!python --version
!orthrus --version

## 2. Environment readiness
`orthrus doctor` checks which optional capabilities are present. It performs **no** network access.

In [ ]:
!orthrus --no-banner doctor

## 3. Start the bundled practice target
A deliberately-vulnerable app that exercises every scanner, bound to `127.0.0.1` only.
We launch it as a background child of the kernel so it survives across cells.

In [ ]:
import subprocess, sys, socket, time

PORT = 8731
target = subprocess.Popen(
    [sys.executable, "tests/integration/reflecting_target.py", str(PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
for _ in range(50):                                   # wait until it accepts connections
    try:
        with socket.create_connection(("127.0.0.1", PORT), timeout=0.5):
            print(f"✅ practice target listening on http://127.0.0.1:{PORT} (pid {target.pid})")
            break
    except OSError:
        time.sleep(0.2)
else:
    raise RuntimeError("practice target did not start")

## 4. Run the full scan
Recon → scan → confirm → report. We drive ORTHRUS as a subprocess (not by importing it) so
its asyncio loop doesn't collide with Colab's. `--no-browser` because the headless-browser
extras aren't installed by default here (see the optional cell at the end to enable them).

In [ ]:
import subprocess
os.makedirs("reports", exist_ok=True)
cmd = [
    "orthrus", "--no-banner", "scan",
    "-t", "http://127.0.0.1:8731",
    "--aggressive", "--no-browser",
    "--crawl-depth", "3", "--max-pages", "50",
    "-o", "reports/colab.json", "--format", "json",
]
print("running:", " ".join(cmd), "\n")
subprocess.run(cmd, check=True)

## 5. Read the results

In [ ]:
import json
data = json.load(open("reports/colab.json"))
s = data["summary"]
print(f"total findings: {s['total']}   confirmed: {s['confirmed']}")
print("by severity:", dict(s["counts"]))
print("-" * 78)
for f in sorted(data["findings"], key=lambda x: x["severity"]):
    print(f"[{f['severity']:<8}] {f['confidence']:<10} {f['vuln_type']:<22} {f['url']}")

## 6. (Optional) Render the themed report UI as a PNG
Installs the browser extra + Chromium, then renders the report into ORTHRUS's terminal UI.

In [ ]:
!pip -q install -e ".[browser]"
!playwright install --with-deps chromium
!python examples/render_report_ui.py reports/colab.json -o reports/colab_ui
from IPython.display import Image
Image("reports/colab_ui.png")

## 7. (Optional) Download the report

In [ ]:
from google.colab import files
files.download("reports/colab.json")

## 8. (Optional) Stop the practice target

---
**Scanning your own authorized target instead?** Replace the target cells with a single
scan that passes an explicit scope, e.g.:

```python
subprocess.run(["orthrus", "--no-banner", "scan",
    "-t", "https://app.you-own.com", "--scope", "*.you-own.com",
    "--no-browser", "-o", "reports/engagement.json", "--format", "json"], check=True)
```

Colab runs from a Google datacenter IP — only scan where that source is authorized.

In [ ]:
target.terminate()
print("stopped practice target")